# Customer Purchase Frequency ETL

## Purpose
Analyze the distribution of days between customer orders to understand purchase frequency patterns and identify customer segments for marketing campaigns.

## Input
* **Source:** `big_data.silver.orders`

## Output
* **Target:** `big_data.gold.vw_customer_purchase_frequency`
* **Refresh:** Real-time (always reflects current Silver data)

## SQL Logic
1. Filter orders with non-null days_since_prior_order
2. Create frequency buckets using CASE (1-7, 8-14, 15-30, 30+ days)
3. COUNT orders per bucket
4. GROUP BY days_range and ORDER BY

In [0]:
%sql
-- Customer Purchase Frequency View
-- Purpose: Analyze purchase frequency patterns with 4 frequency buckets

CREATE OR REPLACE VIEW big_data.gold.vw_customer_purchase_frequency AS
SELECT 
  CASE
    WHEN days_since_prior_order <= 7 THEN '1-7 days'
    WHEN days_since_prior_order <= 14 THEN '8-14 days'
    WHEN days_since_prior_order <= 30 THEN '15-30 days'
    ELSE '30+ days'
  END AS days_range,
  COUNT(*) AS order_count
FROM big_data.silver.orders
WHERE days_since_prior_order IS NOT NULL
GROUP BY days_range
ORDER BY days_range;

In [0]:
%sql
-- Verify view exists and preview frequency buckets
-- Returns 4 rows (one per frequency bucket)

SELECT * FROM big_data.gold.vw_customer_purchase_frequency;